# 循环神经网络（下）双向 LSTM 与变长序列

本节做两件事：

1. **变长序列** 用 `pad_sequence` + `pack_padded_sequence` 处理，避免 LSTM 在 `<pad>` 上做无意义计算。
2. **双向 LSTM**（`bidirectional=True`）在一个合成的"情感分类"任务上展示比单向 LSTM 更强的判别力。

## 1. 合成情感分类任务（信号集中在序列开头）

- 词表 25 个 token：编号 0 是 `<pad>`，1-3 是 "positive" 词（如 `good/great/love`），4-6 是 "negative" 词（如 `bad/awful/hate`），7-24 是中性 filler 词。
- 每条样本：长度 35-45 的中性序列，**前 3 个位置** 全部替换成同极性的情感词（决定标签）。
- 这样设计让"信号"出现在序列**开头**，距离最终的隐状态 35+ 步——这正是单向 LSTM 的长程依赖弱点。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

VOCAB = 25; PAD = 0; POS = [1, 2, 3]; NEG = [4, 5, 6]; FILL = list(range(7, VOCAB))

def make_sample(g):
    L = torch.randint(35, 46, (1,), generator=g).item()
    label = torch.randint(0, 2, (1,), generator=g).item()      # 0 or 1
    tokens = [FILL[torch.randint(0, len(FILL), (1,), generator=g).item()] for _ in range(L)]
    sent_pool = POS if label == 1 else NEG
    for i in range(3):                                          # 前 3 个位置放情感词
        tokens[i] = sent_pool[torch.randint(0, 3, (1,), generator=g).item()]
    return torch.tensor(tokens, dtype=torch.long), label

class SentimentDS(Dataset):
    def __init__(self, n, seed):
        g = torch.Generator().manual_seed(seed)
        self.data = [make_sample(g) for _ in range(n)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

ds = SentimentDS(5, seed=0)
for x, y in ds:
    print(f'len={len(x):2d}  label={y}  tokens={x.tolist()}')

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from nndl.runner import RunnerV3


## 2. `collate_fn` 处理变长 + 排序

DataLoader 把不同长度的样本组成一个 batch 时，要：

1. **pad** 到这个 batch 的最大长度（用 `pad_sequence`）
2. **记下原始长度**（`pack_padded_sequence` 需要）

PyTorch 1.x 时代 `pack_padded_sequence` 要求按长度降序排好；当前版本传 `enforce_sorted=False` 就自动处理。

In [ ]:
def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return padded, lengths, torch.tensor(labels)

train_loader = DataLoader(SentimentDS(2000, seed=0), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(SentimentDS(500,  seed=1), batch_size=64, collate_fn=collate)

padded, lengths, labels = next(iter(train_loader))
print(f'padded.shape = {tuple(padded.shape)}')
print(f'lengths      = {lengths[:8].tolist()} ...')
print(f'labels       = {labels[:8].tolist()} ...')

## 3. 模型：单向 vs 双向 LSTM

PyTorch `nn.LSTM(bidirectional=True)` 输出维度是 `2 * hidden`（前向 + 后向拼接）。我们取 `h_n` 在最后一层的 forward + backward 隐状态拼接做分类特征。

In [ ]:
class SentClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, embed=32, hidden=64, bidirectional=False):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed, padding_idx=PAD)
        self.lstm = nn.LSTM(embed, hidden, batch_first=True, bidirectional=bidirectional)
        n_dir = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden * n_dir, 2)
        self.bidirectional = bidirectional

    def forward(self, padded, lengths):
        e = self.emb(padded)
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        # h: [num_layers * n_dir, B, hidden]，最后一层的方向排在末尾
        # 用负索引让代码对 num_layers > 1 也成立
        if self.bidirectional:
            feat = torch.cat([h[-2], h[-1]], dim=1)      # 最后一层 forward + backward
        else:
            feat = h[-1]                                  # 最后一层
        return self.fc(feat)

print('uni  params :', sum(p.numel() for p in SentClassifier(bidirectional=False).parameters()))
print('bi   params :', sum(p.numel() for p in SentClassifier(bidirectional=True).parameters()))

## 4. 训练比较

In [ ]:
from nndl import accuracy

def train(model, epochs=8, lr=3e-3):
    opt = optim.Adam(model.parameters(), lr=lr)
    runner = RunnerV3(model, opt, nn.CrossEntropyLoss(),
                      metric_fn=accuracy, higher_is_better=True)
    # train_loader 每个 batch 是 (padded, lengths, labels) 三元组——
    # RunnerV3 用 *inputs, y = batch 解包，会自动把前两个当成模型输入。
    runner.fit(train_loader, dev_loader, num_epochs=epochs, log_every=None,
               grad_clip_norm=1.0)
    return runner.history['dev_metric']

torch.manual_seed(0)
uni_hist = train(SentClassifier(bidirectional=False))
torch.manual_seed(0)
bi_hist  = train(SentClassifier(bidirectional=True))

for ep, (u, b) in enumerate(zip(uni_hist, bi_hist), 1):
    print(f'epoch {ep}: uni={u:.4f}  bi={b:.4f}')

In [ ]:
plt.plot(uni_hist, '-o', label='unidirectional')
plt.plot(bi_hist,  '-s', label='bidirectional')
plt.xlabel('epoch'); plt.ylabel('dev accuracy'); plt.ylim(0.4, 1.05); plt.grid(alpha=0.3); plt.legend(); plt.show()

**观察**：

- **单向 LSTM 卡在 ~50%（随机水平）**：信号在开头 3 位，序列长 35-45——前向流到达最终隐状态时，开头的信息基本已经被覆盖。
- **双向 LSTM ≈ 100%**：后向流最后看到的恰好是开头的情感词，毫无衰减压力。
- 这是双向架构的典型受益场景。实际 NLP 里 NER / POS tagging / 阅读理解等任务也类似——每个位置的判断都依赖上下文两个方向的信息。

## 5. 关于 `pack_padded_sequence` 的几个易踩点

- `lengths` 参数必须在 **CPU** 上（即使数据在 GPU）。
- 设 `enforce_sorted=False`，否则要先按长度降序排再恢复——很麻烦。
- 解包：`out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)` 拿回 `[B, L, hidden*n_dir]`。
- `padding_idx=PAD` 让 `nn.Embedding` 对 PAD token 的梯度归零，参数不会被 PAD 拖坏。

## 6. 实践：基于双向 LSTM 的真实 IMDB 文本分类

前面用合成情感任务把双向 LSTM 的长程依赖优势讲清楚了。本节把同一套思路落到真实数据上——经典的 IMDB 影评数据集，搭一个端到端的情感二分类管线，体会真实文本任务里 tokenize、构表、截断补齐、掩码汇聚等工程细节。

> **运行前提**：IMDB 数据集（约 80MB）需要在你自己的环境里准备好，本仓库不附带。下面的数据加载、训练、评价、预测 cell 都用 `DATASET_DIR` 是否存在做了保护：缺数据时只打印提示、不报错；备齐数据后即可整段跑通。模型结构与汇聚算子部分（`AveragePooling` / `Model_BiLSTM_FC`）则用一个极小的合成 batch 做了 forward 自测，不依赖 IMDB 就能验证形状与掩码逻辑。

### 6.1 数据处理

IMDB 是常见的二分类影评数据集：评分 $\geqslant 7$ 标为积极、$\leqslant 4$ 标为消极，训练集和测试集各 25 000 条。官方目录结构如下：

```
|--train/
    |--neg   # 消极
    |--pos   # 积极
    |--unsup # 无标签（本任务用不到）
|--test/
    |--neg
    |--pos
```

LSTM 只能消化数值张量，无法直接读文本，所以第一步要把词转成整数 ID：先准备一张**词表**（vocabulary）把每个词映射成 ID，再走嵌入层查表得到词向量。词表里留一个特殊符号 `[UNK]` 兜底，遇到表外词一律用它代替；占位符 `[PAD]` 固定放在第 0 位（与 `nn.Embedding(padding_idx=0)` 对齐）。

| 词 | ID |
|---|---|
| `[PAD]` | 0 |
| `[UNK]` | 1 |
| the | 2 |
| a | 3 |
| $\cdots$ | $\cdots$ |

In [ ]:
import os
import gzip
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from functools import partial

# IMDB 数据用 practice-in-paddle/dataset/imdb 的 gzip 格式：每个 split 一个
# <split>.txt.gz，每行 `label\ttext`（label 0/1、已小写）；外加 vocab.txt.gz。
# 在若干候选位置里找数据目录，找不到则跳过后续 IMDB cell。
_IMDB_CANDIDATES = [
    os.path.join("..", "datasets", "imdb"),
    os.path.join("..", "..", "..", "practice-in-paddle", "dataset", "imdb"),
    os.path.join("dataset", "imdb"),
]
DATASET_DIR = next((p for p in _IMDB_CANDIDATES
                    if os.path.isfile(os.path.join(p, "train.txt.gz"))), _IMDB_CANDIDATES[0])
HAS_IMDB = (os.path.isfile(os.path.join(DATASET_DIR, "train.txt.gz"))
            and os.path.isfile(os.path.join(DATASET_DIR, "vocab.txt.gz")))


def load_imdb_data(data_path):
    """读取 gzip 版 IMDB，返回 (train_data, dev_data, test_data)。

    每个 split 是 <data_path>/{train,dev,test}.txt.gz，每行 `label\ttext`；
    每条样本输出为 (评论文本, 标签) 二元组。与 Stanford aclImdb_v1 同一份数据，
    只是预处理成了逐行 + gzip + 现成 dev split 和词表。
    """
    def _read_split(split):
        examples = []
        with gzip.open(os.path.join(data_path, f"{split}.txt.gz"), "rt", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                label, text = line.split("\t", 1)
                examples.append((text, int(label)))
        return examples

    return _read_split("train"), _read_split("dev"), _read_split("test")


def load_vocab(vocab_path):
    """gzip 词表：每行一个词，行号即 ID（第 0 行 [PAD]，第 1 行 [UNK]）。"""
    word2id_dict = {}
    with gzip.open(vocab_path, "rt", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            word2id_dict[line.rstrip("\n")] = idx
    return word2id_dict


if HAS_IMDB:
    train_data, dev_data, test_data = load_imdb_data(DATASET_DIR)
    word2id_dict = load_vocab(os.path.join(DATASET_DIR, "vocab.txt.gz"))
    print("数据目录：", DATASET_DIR)
    print("训练集样本数：", len(train_data), " 验证集：", len(dev_data), " 测试集：", len(test_data))
    print("词表大小：", len(word2id_dict))
    print("数据示例：", (train_data[1][0][:60] + "...", train_data[1][1]))
else:
    print("[需 IMDB 数据] 未在以下位置找到 train.txt.gz/vocab.txt.gz：", _IMDB_CANDIDATES,
          "—— 准备好 practice-in-paddle/dataset/imdb，或改 DATASET_DIR 后即可解锁后续 IMDB cell。")


把 IMDB 封装为 `torch.utils.data.Dataset`，沿用前面处理数字序列时的流程：`words_to_id` 依据词表逐词查表，命中取对应 ID，未命中统一用 `[UNK]` 的 ID 代替。

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, examples, word2id_dict):
        super().__init__()
        # 词表：把词转成词表索引
        self.word2id_dict = word2id_dict
        self.examples = self.words_to_id(examples)

    def words_to_id(self, examples):
        tmp_examples = []
        unk_id = self.word2id_dict["[UNK]"]
        for seq, label in examples:
            # 词 -> ID；表外词用 [UNK] 兜底
            seq = [self.word2id_dict.get(word, unk_id) for word in seq.split(" ")]
            tmp_examples.append([seq, int(label)])
        return tmp_examples

    def __getitem__(self, idx):
        seq, label = self.examples[idx]
        return seq, label

    def __len__(self):
        return len(self.examples)

if HAS_IMDB:
    train_set = IMDBDataset(train_data, word2id_dict)
    dev_set = IMDBDataset(dev_data, word2id_dict)
    test_set = IMDBDataset(test_data, word2id_dict)
    print("训练集样本数：", len(train_set))
    print("样本示例（前 20 个 token id）：", train_set[0][0][:20], "... label =", train_set[0][1])

有了 `Dataset` 还需要 `DataLoader` 做批量采样。文本任务比定长任务多两步预处理：

1. **长度限制**：评论长短不一，过长样本拖慢训练并稀释信号，先截到 `max_seq_len` 以内。
2. **长度补齐**：同一批序列必须等长才能拼成张量，短序列用 `[PAD]` 补到本批最长长度。

这两步由自定义的 `collate_fn` 承担。填充会引入一批无意义的 `[PAD]` 位置，下游需要绕开它们，所以 `collate_fn` 同时返回一个 `seq_lens` 记录每条序列的真实长度。配合 `RunnerV3` 的解包惯例 `*inputs, y = batch`，这里直接以「序列张量、长度张量、标签张量」三元组返回，前两项会自动送进 `forward`。

> 注意：合成任务那节用的是 `pad_sequence`（输入已是 `[L]` 的 tensor）；这里 `IMDBDataset` 返回的是 Python `list[int]`，所以截断/补齐都在 list 上做，最后再 `torch.tensor`。

In [ ]:
def collate_fn(batch_data, pad_val=0, max_seq_len=256):
    seqs, seq_lens, labels = [], [], []
    max_len = 0
    for seq, label in batch_data:
        # 截断到 max_seq_len
        seq = seq[:max_seq_len]
        seqs.append(seq)
        seq_lens.append(len(seq))
        labels.append(label)
        max_len = max(max_len, len(seq))
    # 补齐到本批最长长度
    for i in range(len(seqs)):
        seqs[i] = seqs[i] + [pad_val] * (max_len - len(seqs[i]))
    return torch.tensor(seqs), torch.tensor(seq_lens), torch.tensor(labels)

max_seq_len = 256
batch_size = 128
if HAS_IMDB:
    imdb_collate = partial(collate_fn, pad_val=word2id_dict.get("[PAD]"), max_seq_len=max_seq_len)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              drop_last=False, collate_fn=imdb_collate)
    dev_loader = DataLoader(dev_set, batch_size=batch_size, shuffle=False,
                            drop_last=False, collate_fn=imdb_collate)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                             drop_last=False, collate_fn=imdb_collate)
    _seqs, _lens, _labels = next(iter(train_loader))
    print("padded.shape =", tuple(_seqs.shape), " seq_lens[:8] =", _lens[:8].tolist())

### 6.2 模型构建

本实践的模型分四段（对应书图 `rnn-biltm-avg-fc`）：

1. **嵌入层**：整数 ID 序列查表得到词向量序列，直接用 `nn.Embedding(num_embeddings, embedding_dim, padding_idx=0)`。`padding_idx` 指向 `[PAD]`，该位置输出固定为零、梯度不回传，避免填充位扰动训练。
2. **双向 LSTM 层**：在词向量序列上同时跑正向和反向 LSTM，两端隐状态拼接，直接用 `nn.LSTM(..., bidirectional=True)`。用 `pack_padded_sequence` 打包变长序列，LSTM 内部会跳过 `[PAD]`。
3. **汇聚层**：对各时刻隐状态按真实长度做平均，把整条序列压成一个句向量。
4. **线性层**：把句向量映射到类别 logits，用 `nn.Linear`。

与合成任务那节取「末步隐状态」不同，这里用的是**掩码平均汇聚**：`pack/pad` 还原后 `[PAD]` 位置是零向量，`AveragePooling` 再按 `seq_lens` 生成掩码、只对真实位置求和并归一化。为了让汇聚层与序列模型解耦、可独立替换，这里仍做一次显式掩码而不是依赖打包语义。

In [ ]:
class AveragePooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, sequence_output, sequence_length):
        sequence_length = sequence_length.unsqueeze(-1).to(torch.float32)
        # 按真实长度生成掩码，屏蔽 [PAD] 位置
        max_len = sequence_output.shape[1]
        mask = torch.arange(max_len) < sequence_length
        mask = mask.to(torch.float32).unsqueeze(-1)
        # 掩蔽 [PAD] 部分
        sequence_output = torch.mul(sequence_output, mask)
        # 对真实位置求和，再按真实长度归一化
        batch_mean_hidden = torch.div(torch.sum(sequence_output, dim=1), sequence_length)
        return batch_mean_hidden

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Model_BiLSTM_FC(nn.Module):
    def __init__(self, num_embeddings, input_size, hidden_size, num_classes=2):
        super().__init__()
        self.num_embeddings = num_embeddings   # 词表大小
        self.input_size = input_size           # 词向量维度
        self.hidden_size = hidden_size          # LSTM 隐藏单元数
        self.num_classes = num_classes
        self.embedding_layer = nn.Embedding(num_embeddings, input_size, padding_idx=0)
        # batch_first=True 表示输入形状是 [batch, length, embedding]
        self.lstm_layer = nn.LSTM(input_size, hidden_size,
                                  batch_first=True, bidirectional=True)
        self.average_layer = AveragePooling()
        self.output_layer = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, input_ids, sequence_length):
        inputs_emb = self.embedding_layer(input_ids)
        # 打包变长序列，LSTM 内部跳过 [PAD]；lengths 需在 CPU 上
        packed = pack_padded_sequence(inputs_emb, sequence_length.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm_layer(packed)
        # 解包回 [batch, length, 2*hidden]，[PAD] 处为 0
        sequence_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        batch_mean_hidden = self.average_layer(sequence_output, sequence_length)
        logits = self.output_layer(batch_mean_hidden)
        return logits

下面用一个极小的合成 batch（不依赖 IMDB）对 `AveragePooling` 与 `Model_BiLSTM_FC` 做 forward 自测，确认形状与掩码逻辑正确。

迁移过程中改掉的几处「书 -> 可运行 notebook」差异（book-ism）：

- 书里 `from utils.data import load_imdb_data, load_vocab`、`from utils.data import load_vocab`——该 `utils.data` 模块未随 notebook 提供，已内联重写两个读取函数。
- 书里 `IMDBDataset.__getitem__` 用了 `enumerate` 但没用到下标，已简化为直接解包。
- `nndl` 的导入沿用本 notebook 顶部已 `sys.path.insert` 父目录后的 `from nndl ...`。
- 真实数据/训练/评价/预测都加了 `HAS_IMDB` 保护，缺数据时打印「需用户环境 + IMDB 数据」提示而非报错。

In [ ]:
# forward 自测：极小合成 batch，无需 IMDB
_toy_vocab = {"[PAD]": 0, "[UNK]": 1, "great": 2, "movie": 3, "bad": 4, "this": 5}
_toy = [("this great movie", "1"), ("this bad zzz unknownword", "0")]
_ds = IMDBDataset(_toy, _toy_vocab)
_cf = partial(collate_fn, pad_val=0, max_seq_len=256)
_seqs, _lens, _labels = _cf([_ds[0], _ds[1]])
print("toy ids[1] =", _ds[1][0], "(zzz/unknownword -> [UNK]=1)")
print("padded =", _seqs.tolist(), " seq_lens =", _lens.tolist())

# AveragePooling 掩码：全 1 隐状态的掩码均值必为 1，与真实长度无关
_pool = AveragePooling()
assert torch.allclose(_pool(torch.ones(2, 4, 3), torch.tensor([2, 4])), torch.ones(2, 3))

torch.manual_seed(0)
_m = Model_BiLSTM_FC(num_embeddings=len(_toy_vocab), input_size=8, hidden_size=8)
_logits = _m(_seqs, _lens)
print("logits.shape =", tuple(_logits.shape))
assert _logits.shape == (2, 2)
print("forward 自测通过 [OK]")

### 6.3 模型训练

继续沿用 `RunnerV3`：配齐嵌入维度、隐状态维度、优化器、损失、评价指标，训练 3 个 epoch。为防止梯度爆炸卷土重来，同时打开 `grad_clip_norm=1.0`。验证集上表现最佳的检查点会被保存，后续评价基于它而非最后一轮。

> 真实 IMDB（25 000 条、序列截到 256、`hidden=256`）在 CPU 上训练较慢，建议有 GPU 时给 `RunnerV3(..., device='cuda')`。下面用 `HAS_IMDB` 保护，缺数据时不执行。

In [ ]:
from nndl import RunnerV3, accuracy

if HAS_IMDB:
    num_embeddings = len(word2id_dict)   # 词表大小
    input_size = 256                     # 词向量维度
    hidden_size = 256                    # LSTM 隐状态维度
    model = Model_BiLSTM_FC(num_embeddings, input_size, hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
    loss_fn = nn.CrossEntropyLoss()
    runner = RunnerV3(model, optimizer, loss_fn, metric_fn=accuracy)
    os.makedirs("./checkpoints", exist_ok=True)
    runner.fit(train_loader, dev_loader, num_epochs=3, log_every=1,
               best_path="./checkpoints/best_imdb_bilstm.pt", grad_clip_norm=1.0)
else:
    print("[需用户环境 + IMDB 数据] 跳过训练。备齐 ./dataset/ 后重跑本 cell。")

In [ ]:
# 训练损失与验证准确率曲线（对应书图 rnn-lstm-practice-loss）
if HAS_IMDB:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(runner.history["train_loss"], "-o", label="train loss")
    ax1.plot(runner.history["dev_loss"],   "-s", label="dev loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(runner.history["dev_metric"], "-^", color="green", label="dev accuracy")
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


### 6.4 模型评价与预测

加载验证集上效果最好的检查点，在测试集上评估准确率；再随便给一句话，让训练好的模型判定情感极性。预测流程复用训练时的分词与截断规则，把单句包装成 `[1, L]` 的批张量。

> 书里参考实现的测试准确率约为 0.86。

In [ ]:
id2label = {0: "消极情绪", 1: "积极情绪"}

if HAS_IMDB:
    # 加载验证集最优检查点（RunnerV3.load 读 state_dict）
    runner.load("./checkpoints/best_imdb_bilstm.pt")
    test_acc, _ = runner.evaluate(test_loader)
    print(f"Evaluate on test set, Accuracy: {test_acc:.5f}")

    # 单句预测
    text = "this movie is so great. I watched it three times already"
    tokens = [word2id_dict.get(w, word2id_dict["[UNK]"]) for w in text.split(" ")]
    tokens = tokens[:max_seq_len]
    sequence_length = torch.tensor([len(tokens)], dtype=torch.int64)
    tokens = torch.tensor(tokens, dtype=torch.int64).unsqueeze(0)
    logits = runner.predict(tokens, sequence_length)
    pred_label = id2label[torch.argmax(logits, dim=-1).item()]
    print("Label: ", pred_label)
else:
    print("[需用户环境 + IMDB 数据] 跳过评价与预测。备齐 ./dataset/ 后重跑本 cell。")

## 小结

- **变长序列**：`pad_sequence` + `pack_padded_sequence` 是标准做法；LSTM 跳过 PAD 位置的计算。
- **双向 LSTM**：`bidirectional=True`，输出维度 `hidden * 2`；要从开头和末尾都提取信息的任务上受益明显。
- 实际文本分类（IMDB / AG_News 等）流程基本相同：tokenize → vocab + pad → embedding → bi-LSTM → linear → CE loss。
- 下一步（chap7）：网络优化与正则化——更系统地讲 lr、optimizer、weight decay、dropout 等。